In [3]:
# Databricks notebook source
import requests
import time

import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [ ]:
# COMMAND ----------

dbutils.widgets.text("output_catalog", "poc")
dbutils.widgets.text("output_schema", "testing")

dbutils.widgets.text("checkpoint_volume", "volume")

In [ ]:
# COMMAND ----------

# store service principal token in secret scope, retrieve from there
access_token = dbutils.secrets.get(scope = "", key = "") 

In [ ]:
# COMMAND ----------

output_catalog = dbutils.widgets.get("output_catalog")
output_schema = dbutils.widgets.get("output_schema")
output_table = "task_runs"

checkpoint_volume = dbutils.widgets.get("checkpoint_volume")

job_runs_table = f"{output_catalog}.{output_schema}.job_runs"
task_runs_table = f"{output_catalog}.{output_schema}.{output_table}"

checkpoint_volume = f"/Volumes/{output_catalog}/{output_schema}/{checkpoint_volume}/{output_table}/checkpoint"

print(job_runs_table)
print(task_runs_table)
print(checkpoint_volume)

In [ ]:
# COMMAND ----------

def job_runs_api(access_token, runs_endpoint, max_retries = 600):
    """ 
    Returns UDF for job runs API requests.
    """
    
    @F.udf("string")
    def job_runs_api_udf(workspace_url, run_id):
        
        url = f"https://f{workspace_url}/api/2.2/jobs/runs/{runs_endpoint}?run_id={run_id}"
        headers = {
            "Authorization": f"Bearer {access_token}"
        }

        retries = 0

        while retries < max_retries:

            response = requests.get(url, headers=headers)
            
            if response.status_code == 200 or response.status_code == 400:
                # exports aren't always present and if they are not present can return a 400
                return response.text
            
            retries += 1
            time.sleep(1)

        raise ConnectionError(
            f"""
            {runs_endpoint} request ({run_id}) unsuccessful after {max_retries} attempts, 
            status code {response.status_code}
            """
            )
        
    return job_runs_api_udf

In [ ]:

# COMMAND ----------

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {task_runs_table} (
        task_run_id STRING,
        job_run_id STRING,
        job_id STRING,
        workspace_url STRING,
        task_run_output_json STRING,
        task_run_output VARIANT,
        task_run_export_json STRING,
        task_run_export VARIANT,
        updated_ts TIMESTAMP
    )
    CLUSTER BY (task_run_id, job_run_id, job_id)
    TBLPROPERTIES (
        delta.enableChangeDataFeed = true,
        delta.enableDeletionVectors = true
        )
    """
)

In [ ]:
# COMMAND ----------

def for_each_batch(df, id):

    df = df.filter(df._change_type.isin(["insert", "update"]))

    # Pull task runs out of job runs

    df = df.select(
        "*", 
        F.explode(F.try_variant_get(df.job_run, "$.tasks", "ARRAY<VARIANT>")).alias("task")
        )

    df = df.select(
        F.try_variant_get(df.task, "$.run_id", "STRING").alias("task_run_id"),
        df.run_id.alias("job_run_id"), 
        df.job_id,
        df.workspace_url
    )

    df = df.distinct()

    # Get API responses for output and export

    df = df.withColumn(
        "task_run_output_json", 
        job_runs_api(access_token, "get-output")(df.workspace_url, df.task_run_id)
        )
    df = df.withColumn("task_run_output", F.try_parse_json(df.task_run_output_json))
    df = df.withColumn(
        "task_run_export_json", 
        job_runs_api(access_token, "export")(df.workspace_url, df.task_run_id)
        )
    df = df.withColumn("task_run_export", F.try_parse_json(df.task_run_export_json))
    df = df.withColumn("updated_ts", F.current_timestamp())

    # Execute merge into output table

    delta_output_table = DeltaTable.forName(spark, task_runs_table)

    (
        delta_output_table
            .alias('target')
            .merge(
                df.alias('updates'),
                'target.task_run_id = updates.task_run_id'
                )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
    )

In [ ]:
# COMMAND ----------

df = (
    spark.readStream
    .option("readChangeFeed", "true")
    .table(job_runs_table)
)

# COMMAND ----------

query = (
    df.writeStream
    .queryName(task_runs_table)
    .foreachBatch(for_each_batch)
    .option("checkpointLocation", checkpoint_volume)
    .trigger(availableNow=True)
    .start()
)